# video-app lab 08 - first clip, version-proof
Notebook 07 hit a version mismatch: this diffusers build has no `pipe.enable_vae_slicing()` / no `dtype=` kwarg.
Every call below is guarded: it tries the new API, falls back to the old one, and never dies on a cosmetic warning.
Stack is unchanged and free: SD1.5 base + AnimateDiff v1-5-2 + Lightning distilled motion weights.


In [ ]:
import os, shutil, psutil, torch
print(f'RAM {psutil.virtual_memory().used/1024**3:.1f}/{psutil.virtual_memory().total/1024**3:.1f} GB | disk {shutil.disk_usage("/").free/1024**3:.0f} GB | cuda {torch.cuda.is_available()}')
if not torch.cuda.is_available(): raise SystemExit('STOP: no GPU. Runtime > Change runtime type > T4 GPU + High-RAM > Save.')
print('GPU', torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/1024**3,1), 'GB')
import diffusers; print('diffusers', diffusers.__version__)


In [ ]:
import gc, time, torch
from diffusers import AnimateDiffPipeline, MotionAdapter, EulerDiscreteScheduler
DTYPE = torch.float16
def load(cls, repo, **kw):
    kw = dict(kw, low_cpu_mem_usage=True)
    try: return cls.from_pretrained(repo, dtype=DTYPE, **kw)
    except TypeError: return cls.from_pretrained(repo, torch_dtype=DTYPE, **kw)
def soft(obj, name):
    try: getattr(obj, name)(); print('ok', name)
    except Exception as e: print('skip', name, '-', type(e).__name__)

adapter = load(MotionAdapter, 'guoyww/animatediff-motion-adapter-v1-5-2')
pipe = load(AnimateDiffPipeline, 'emilianJR/epiCRealism', motion_adapter=adapter, use_safetensors=True)
pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config, timestep_spacing='trailing', beta_schedule='linear')
soft(pipe, 'enable_vae_slicing'); soft(pipe, 'enable_attention_slicing')
try: pipe.vae.enable_slicing()
except Exception as e: print('vae slicing via vae:', type(e).__name__)
pipe.to('cuda'); gc.collect()
print('loaded | VRAM', round(torch.cuda.memory_allocated()/1024**3,2), 'GB')


In [ ]:
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
STEPS, GUID = 25, 7.5
try:
    sd = load_file(hf_hub_download('ByteDance/AnimateDiff-Lightning','animatediff_lightning_4step_diffusers.safetensors'))
    sd = {('unet.'+k): v.to(DTYPE) for k, v in sd.items()}
    res = pipe.unet.load_state_dict(sd, strict=False)
    print('missing', len(res.missing_keys), 'unexpected', len(res.unexpected_keys))
    if len(res.unexpected_keys) == 0:
        STEPS, GUID = 4, 1.0; print('LIGHTNING OK -> 4 steps')
except Exception as e:
    print('lightning skipped:', type(e).__name__, str(e)[:150])
pipe.to('cuda')


In [ ]:
from diffusers.utils import export_to_video
from IPython.display import Video, display
PROMPT = ('cinematic photo of a red fox trotting through fresh snow in a pine forest, '
          'morning light rays, shallow depth of field, steady camera, 35mm film')
t0 = time.time()
frames = pipe(prompt=PROMPT, negative_prompt='bad quality, blurry, distorted, watermark, text',
              num_frames=16, height=512, width=512, guidance_scale=GUID, num_inference_steps=STEPS,
              generator=torch.Generator('cpu').manual_seed(42)).frames[0]
print('GENERATED', STEPS, 'steps in', round(time.time()-t0), 's | peak VRAM', round(torch.cuda.max_memory_allocated()/1024**3,2), 'GB')
export_to_video(frames, '/content/clip_08_fox.mp4', fps=8)
print('file MB', round(os.path.getsize('/content/clip_08_fox.mp4')/1024**2,1))
display(Video('/content/clip_08_fox.mp4', embed=True, width=512))
from google.colab import files; files.download('/content/clip_08_fox.mp4')
